In [ ]:


import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
pd.set_option('display.max_columns', None)

PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Products' / 'Chamber Study Missions' / '2026'
PATH_I = Path(r'I:\Projects\Josh\Regional Monitoring')


def re_remove_post(x, exp = ' '):
    try:
        return x.split(exp, 1)[0]
    except Exception as e:
        e
        return x


In [ ]:


EXPORT=True


file_pop = PATH_SP / 'Pop_3 MPO ACS5_prt_SEMCOG.xlsx'
df_pop = pd.read_excel(file_pop, sheet_name='MPO')
df_pop.head()



In [ ]:


min_year = 2014
max_year = 2024

df_pop2 = df_pop.copy()
df_pop2 = df_pop2[(df_pop2['Year'] == max_year) | (df_pop2['Year'] == min_year)]
df_pop2 = df_pop2.drop(['Variable'], axis=1)

df_pop2 = df_pop2.pivot_table(index=['MPO', 'Year'], columns='Race/Ethnicity', values='Population').reset_index()

eths = list(df_pop['Race/Ethnicity'].unique())
for eth in eths:
    col_pct_change = f'{eth}_pct_change'
    df_pop2[col_pct_change] = df_pop2[eth].pct_change()
    df_pop2.loc[df_pop2[col_pct_change] == np.inf, col_pct_change] = np.nan
    df_pop2 = df_pop2.drop(eth, axis=1)

df_pop2.loc[df_pop2['Year'] == min_year, col_pct_change] = np.nan
df_pop2 = df_pop2[df_pop2['Year'] == max_year]

df_pop2['Year'] = f'{min_year} to {max_year}'

display(df_pop2)


if EXPORT:
    with pd.ExcelWriter(file_pop, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pop2.to_excel(writer, sheet_name='PctPopGrowth', index=False)


